In [3]:
# Install einops
!pip install einops

# Then re-run the imports
import os, time, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from albumentations import Compose, Resize, Normalize, HorizontalFlip, RandomBrightnessContrast, ShiftScaleRotate
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score
import timm
import cv2
from einops import rearrange, repeat

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
cv2.setNumThreads(0)

print("="*70)
print("PHASE 7: ADVANCED TECHNIQUES - ALL COMBINED")
print("="*70)
print(f"Device: {device}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*70)


PHASE 7: ADVANCED TECHNIQUES - ALL COMBINED
Device: cuda
CUDA: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU Memory: 8.6 GB


In [4]:
class AdvancedMultiModalDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, transform, use_metadata=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.use_metadata = use_metadata
        self.labels = ['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema',
                       'Emphysema','Fibrosis','Effusion','Pneumonia','Pleural_Thickening',
                       'Cardiomegaly','Nodule','Mass','Hernia']
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, i):
        try:
            row = self.df.iloc[i]
            
            # Load image
            img_path = None
            for j in range(1,13):
                p = os.path.join(self.img_dir, f'images_{j:03d}/images', row['Image Index'])
                if os.path.exists(p): img_path = p; break
            
            img = np.array(Image.open(img_path).convert('RGB'))
            img = self.transform(image=img)['image']
            
            # Extract metadata
            if self.use_metadata:
                age = float(row['Patient Age']) / 100.0  # Normalize [0,1]
                gender = 1.0 if row['Patient Gender'] == 'M' else 0.0
                view_pa = 1.0 if row['View Position'] == 'PA' else 0.0
                
                # Additional features
                follow_up = float(row.get('Follow-up #', 0)) / 10.0  # Normalize
                
                metadata = torch.FloatTensor([age, gender, view_pa, follow_up])
            else:
                metadata = torch.zeros(4)
            
            # Labels
            y = np.zeros(len(self.labels), dtype=np.float32)
            if row['Finding Labels'] != "No Finding":
                for k, l in enumerate(self.labels):
                    if l in row['Finding Labels']: y[k] = 1
            
            return img, metadata, torch.tensor(y)
        
        except Exception as e:
            return None

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    imgs, metas, labels = zip(*batch)
    return torch.stack(imgs), torch.stack(metas), torch.stack(labels)


In [5]:
class AnatomicalPriorAttention(nn.Module):
    """Disease-specific spatial attention guided by anatomy"""
    def __init__(self, in_channels=768, num_diseases=14):
        super().__init__()
        
        # Learn disease-specific attention maps
        self.disease_attention = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_diseases, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Global context
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.global_fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // 4),
            nn.ReLU(),
            nn.Linear(in_channels // 4, num_diseases),
            nn.Sigmoid()
        )
    
    def forward(self, features):
        # features: [batch, channels, H, W]
        batch_size, C, H, W = features.shape
        
        # Spatial attention maps per disease
        spatial_attn = self.disease_attention(features)  # [batch, 14, H, W]
        
        # Global disease relevance
        global_feat = self.global_pool(features).squeeze(-1).squeeze(-1)
        global_attn = self.global_fc(global_feat)  # [batch, 14]
        
        # Combine spatial and global
        global_attn = global_attn.unsqueeze(-1).unsqueeze(-1)  # [batch, 14, 1, 1]
        combined_attn = spatial_attn * global_attn  # [batch, 14, H, W]
        
        # Average attention across diseases
        avg_attn = combined_attn.mean(dim=1, keepdim=True)  # [batch, 1, H, W]
        
        # Apply attention
        attended_features = features * (1 + avg_attn)  # Residual attention
        
        return attended_features, combined_attn


In [6]:
class UltimateChestXrayModel(nn.Module):
    """
    Combines:
    1. MaxViT hybrid architecture (CNN + Transformer)
    2. Anatomical prior attention
    3. Multi-modal fusion (image + metadata)
    4. Disease-specific heads
    """
    def __init__(self, num_classes=14, metadata_dim=4, use_dinov2=False):
        super().__init__()
        
        self.use_dinov2 = use_dinov2
        
        if use_dinov2:
            # Option A: DINOv2 backbone (better for medical images)
            print("Loading DINOv2 ViT-B/14...")
            self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
            self.feature_dim = 768
            
            # Freeze backbone initially
            for param in self.backbone.parameters():
                param.requires_grad = False
            
            # Adapter for feature extraction
            self.feature_adapter = nn.Identity()
            
        else:
            # Option B: MaxViT hybrid (CNN + Transformer blocks)
            print("Loading MaxViT-Base...")
            self.backbone = timm.create_model(
                'maxvit_base_tf_224.in1k',
                pretrained=True,
                features_only=True,
                out_indices=[3]  # Last feature map
            )
            self.feature_dim = 512  # MaxViT feature dimension
            self.feature_adapter = nn.Conv2d(512, 768, 1)  # Adapt to standard dim
        
        # Anatomical attention
        self.anatomical_attention = AnatomicalPriorAttention(
            in_channels=768,
            num_diseases=num_classes
        )
        
        # Metadata encoder
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256)
        )
        
        # Global pooling
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Fusion and classification
        fusion_dim = 768 + 256  # Image features + metadata
        
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, img, metadata):
        batch_size = img.size(0)
        
        # Extract image features
        if self.use_dinov2:
            # DINOv2: [batch, 768]
            img_features = self.backbone(img)
            # Reshape for attention: [batch, 768, 1, 1]
            img_features_spatial = img_features.unsqueeze(-1).unsqueeze(-1)
            img_features_spatial = F.interpolate(
                img_features_spatial, 
                size=(14, 14), 
                mode='bilinear'
            )
        else:
            # MaxViT: [batch, 512, 7, 7]
            img_features_list = self.backbone(img)
            img_features_spatial = img_features_list[0]
            img_features_spatial = self.feature_adapter(img_features_spatial)
        
        # Apply anatomical attention
        attended_features, attention_maps = self.anatomical_attention(img_features_spatial)
        
        # Global pool
        img_global = self.pool(attended_features).flatten(1)  # [batch, 768]
        
        # Encode metadata
        meta_features = self.metadata_encoder(metadata)  # [batch, 256]
        
        # Fuse image and metadata
        combined = torch.cat([img_global, meta_features], dim=1)  # [batch, 1024]
        
        # Final classification
        logits = self.fusion_head(combined)  # [batch, 14]
        
        return logits, attention_maps


In [7]:
extract = "./chest_xray_data/"
data = pd.read_csv(os.path.join(extract, "Data_Entry_2017.csv"))

# Same split for comparison
train_df, val_df, test_df = np.split(
    data.sample(frac=1, random_state=42),
    [int(.7*len(data)), int(.85*len(data))]
)

# Enhanced transforms
tf_train = Compose([
    Resize(224, 224),
    HorizontalFlip(p=0.5),
    ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=10, p=0.3),
    RandomBrightnessContrast(0.2, 0.2, p=0.5),
    Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

tf_val = Compose([
    Resize(224, 224),
    Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

train_ds = AdvancedMultiModalDataset(train_df, extract, tf_train, use_metadata=True)
val_ds = AdvancedMultiModalDataset(val_df, extract, tf_val, use_metadata=True)
test_ds = AdvancedMultiModalDataset(test_df, extract, tf_val, use_metadata=True)

# Balanced sampler
labels = ['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema',
          'Emphysema','Fibrosis','Effusion','Pneumonia','Pleural_Thickening',
          'Cardiomegaly','Nodule','Mass','Hernia']

def create_sampler(df, labels):
    class_counts = np.zeros(len(labels))
    for i, label in enumerate(labels):
        class_counts[i] = df['Finding Labels'].str.contains(label, na=False).sum()
    
    class_weights = 1.0 / (class_counts + 1)
    class_weights = class_weights / class_weights.sum() * len(labels)
    
    sample_weights = []
    for idx, row in df.iterrows():
        labels_present = [i for i, l in enumerate(labels) if l in row['Finding Labels']]
        if labels_present:
            sample_weights.append(max(class_weights[labels_present]))
        else:
            sample_weights.append(1.0)
    
    return torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

sampler = create_sampler(train_df, labels)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=10, sampler=sampler,
                          num_workers=0, collate_fn=safe_collate, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=10, shuffle=False,
                        num_workers=0, collate_fn=safe_collate, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=10, shuffle=False,
                         num_workers=0, collate_fn=safe_collate, pin_memory=False)

print(f"✅ Data loaded:")
print(f"   Train: {len(train_df):,} images ({len(train_loader)} batches)")
print(f"   Val:   {len(val_df):,} images ({len(val_loader)} batches)")
print(f"   Test:  {len(test_df):,} images ({len(test_loader)} batches)")


C:\Users\91850\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
C:\Users\91850\anaconda3\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


✅ Data loaded:
   Train: 78,484 images (7849 batches)
   Val:   16,818 images (1682 batches)
   Test:  16,818 images (1682 batches)


In [9]:
# Choose architecture
USE_DINOV2 = True  # Set False to use MaxViT instead

model = UltimateChestXrayModel(
    num_classes=14,
    metadata_dim=4,
    use_dinov2=USE_DINOV2
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Ultimate Model:")
print(f"   Architecture: {'DINOv2' if USE_DINOV2 else 'MaxViT'} + Anatomical Attn + Multi-Modal")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Class weights
class_counts = np.zeros(14)
for _, row in train_df.iterrows():
    for i, l in enumerate(labels):
        if l in row['Finding Labels']: class_counts[i] += 1

class_weights = len(train_df) / (14 * (class_counts + 1))
class_weights = torch.FloatTensor(class_weights).to(device)

# Focal Loss with label smoothing
class AdvancedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, inputs, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        
        BCE = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE)
        focal = (1 - pt) ** self.gamma * BCE
        
        if self.alpha is not None:
            alpha_t = self.alpha.unsqueeze(0).expand_as(targets)
            focal = alpha_t * focal
        
        return focal.mean()

loss_fn = AdvancedFocalLoss(alpha=class_weights, gamma=2, label_smoothing=0.1)

# Optimizer with different LRs for different components
if USE_DINOV2:
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 5e-6},  # Frozen/very low
        {'params': model.anatomical_attention.parameters(), 'lr': 5e-5},
        {'params': model.metadata_encoder.parameters(), 'lr': 1e-4},
        {'params': model.fusion_head.parameters(), 'lr': 1e-4}
    ], weight_decay=1e-4)
else:
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 1e-5},
        {'params': model.anatomical_attention.parameters(), 'lr': 5e-5},
        {'params': model.metadata_encoder.parameters(), 'lr': 1e-4},
        {'params': model.fusion_head.parameters(), 'lr': 1e-4}
    ], weight_decay=1e-4)

# Scheduler with warmup
from torch.optim.lr_scheduler import OneCycleLR

scheduler = OneCycleLR(
    optimizer,
    max_lr=[5e-6, 5e-5, 1e-4, 1e-4] if USE_DINOV2 else [1e-5, 5e-5, 1e-4, 1e-4],
    epochs=15,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy='cos'
)

scaler = GradScaler(device='cuda')

print("\n✅ Training setup complete")


Loading DINOv2 ViT-B/14...


Using cache found in C:\Users\91850/.cache\torch\hub\facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to C:\Users\91850/.cache\torch\hub\checkpoints\dinov2_vitb14_pretrain.pth


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 330M/330M [28:25<00:00, 203kB/s]



✅ Ultimate Model:
   Architecture: DINOv2 + Anatomical Attn + Multi-Modal
   Total parameters: 87,928,234
   Trainable parameters: 1,347,754

✅ Training setup complete


In [10]:
def train_epoch(model, loader, loss_fn, opt, scaler, scheduler, epoch, log_file):
    model.train()
    total_loss = 0
    labs, preds = [], []
    
    start_time = time.time()
    pbar = tqdm(loader, desc=f"Epoch {epoch:2d} Train", ncols=120, mininterval=1.0)
    
    for batch_idx, batch in enumerate(pbar):
        if batch is None: continue
        img, metadata, y = batch
        img, metadata, y = img.to(device), metadata.to(device), y.to(device)
        
        opt.zero_grad(set_to_none=True)
        
        with autocast(device_type='cuda', dtype=torch.float16):
            logits, attn_maps = model(img, metadata)
            loss = loss_fn(logits, y)
        
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        labs.append(y.detach().cpu().numpy())
        preds.append(torch.sigmoid(logits.detach()).cpu().numpy())
        
        if batch_idx % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            lr = opt.param_groups[-1]['lr']
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'lr': f'{lr:.2e}'})
        
        if batch_idx % 100 == 0:
            with open(log_file, 'a') as f:
                f.write(f"[{time.time():.0f}] Epoch {epoch} | Batch {batch_idx}/{len(loader)} | Loss {loss.item():.4f}\n")
    
    elapsed = time.time() - start_time
    auc = roc_auc_score(np.vstack(labs), np.vstack(preds), average='macro')
    
    return total_loss / len(loader), auc, elapsed

def val_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    labs, preds = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", ncols=120):
            if batch is None: continue
            img, metadata, y = batch
            img, metadata, y = img.to(device), metadata.to(device), y.to(device)
            
            with autocast(device_type='cuda', dtype=torch.float16):
                logits, _ = model(img, metadata)
                loss = loss_fn(logits, y)
            
            total_loss += loss.item()
            labs.append(y.cpu().numpy())
            preds.append(torch.sigmoid(logits).cpu().numpy())
    
    auc = roc_auc_score(np.vstack(labs), np.vstack(preds), average='macro')
    per_class_auc = roc_auc_score(np.vstack(labs), np.vstack(preds), average=None)
    
    return total_loss / len(loader), auc, per_class_auc


In [11]:
log_file = f'phase7_{"dinov2" if USE_DINOV2 else "maxvit"}_ultimate_training.log'
results_file = f'phase7_{"dinov2" if USE_DINOV2 else "maxvit"}_results.json'

with open(log_file, 'w') as f:
    f.write(f"Phase 7 Ultimate Training Started: {time.ctime()}\n")
    f.write(f"Architecture: {'DINOv2' if USE_DINOV2 else 'MaxViT'} + Anatomical + Multi-Modal\n")
    f.write(f"Target: 0.84-0.86 AUC\n\n")

best_auc = 0
patience = 5
patience_counter = 0
results = []

print("\n" + "="*70)
print("🚀 STARTING ULTIMATE MODEL TRAINING")
print("="*70)
print(f"Architecture: {'DINOv2' if USE_DINOV2 else 'MaxViT'} + Anatomical + Multi-Modal")
print(f"Target AUC: 0.84-0.86")
print(f"Start time: {time.strftime('%I:%M %p')}")
print("="*70)

for epoch in range(1, 16):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch}/15")
    print(f"{'='*70}")
    
    tr_loss, tr_auc, tr_time = train_epoch(
        model, train_loader, loss_fn, optimizer, scaler, scheduler, epoch, log_file
    )
    
    val_loss, val_auc, per_class_auc = val_epoch(model, val_loader, loss_fn)
    
    print(f"\n📊 Results:")
    print(f"   Train: Loss={tr_loss:.4f} | AUC={tr_auc:.4f} | Time={tr_time/60:.1f}min")
    print(f"   Val:   Loss={val_loss:.4f} | AUC={val_auc:.4f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f"best_ultimate_{'dinov2' if USE_DINOV2 else 'maxvit'}.pth")
        patience_counter = 0
        
        print(f"\n✅ NEW BEST! AUC={best_auc:.4f}")
        
        if best_auc >= 0.84:
            print(f"🎉 TARGET ACHIEVED! {best_auc:.4f} >= 0.84")
        
        print("\n   Per-Class AUC:")
        for i, label in enumerate(labels):
            print(f"      {label:20s}: {per_class_auc[i]:.4f}")
    else:
        patience_counter += 1
        print(f"   No improvement ({patience_counter}/{patience})")
    
    results.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_auc': tr_auc,
        'val_loss': val_loss,
        'val_auc': val_auc,
        'per_class_auc': per_class_auc.tolist()
    })
    
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    with open(log_file, 'a') as f:
        f.write(f"[{time.time():.0f}] Epoch {epoch} | Train AUC: {tr_auc:.4f} | Val AUC: {val_auc:.4f} | Best: {best_auc:.4f}\n")
    
    if patience_counter >= patience:
        print(f"\n⏹️ Early stopping at epoch {epoch}")
        break
    
    torch.cuda.empty_cache()

print("\n" + "="*70)
print("✅ ULTIMATE MODEL TRAINING COMPLETE")
print("="*70)
print(f"End time: {time.strftime('%I:%M %p')}")
print(f"Best Val AUC: {best_auc:.4f}")
print(f"Model saved: best_ultimate_{'dinov2' if USE_DINOV2 else 'maxvit'}.pth")
print("="*70)



🚀 STARTING ULTIMATE MODEL TRAINING
Architecture: DINOv2 + Anatomical + Multi-Modal
Target AUC: 0.84-0.86
Start time: 10:57 PM

EPOCH 1/15


Epoch  1 Train:   0%|                                                                          | 0/7849 [00:00…

C:\Users\91850\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Validating:   0%|                                                                              | 0/1682 [00:00…


📊 Results:
   Train: Loss=0.1933 | AUC=0.5826 | Time=78.4min
   Val:   Loss=0.1191 | AUC=0.7097

✅ NEW BEST! AUC=0.7097

   Per-Class AUC:
      Atelectasis         : 0.6639
      Consolidation       : 0.7500
      Infiltration        : 0.6340
      Pneumothorax        : 0.7826
      Edema               : 0.8016
      Emphysema           : 0.7593
      Fibrosis            : 0.7177
      Effusion            : 0.7291
      Pneumonia           : 0.6704
      Pleural_Thickening  : 0.7098
      Cardiomegaly        : 0.6614
      Nodule              : 0.5926
      Mass                : 0.6117
      Hernia              : 0.8516

EPOCH 2/15


...

Validating:   0%|                                                                              | 0/1682 [00:00…


📊 Results:
   Train: Loss=0.1050 | AUC=0.8105 | Time=69.3min
   Val:   Loss=0.1094 | AUC=0.7521
   No improvement (3/5)

EPOCH 15/15


Epoch 15 Train:   0%|                                                                          | 0/7849 [00:00…

Validating:   0%|                                                                              | 0/1682 [00:00…


📊 Results:
   Train: Loss=0.1074 | AUC=0.8102 | Time=65.2min
   Val:   Loss=0.1087 | AUC=0.7519
   No improvement (4/5)

✅ ULTIMATE MODEL TRAINING COMPLETE
End time: 12:08 AM
Best Val AUC: 0.7545
Model saved: best_ultimate_dinov2.pth


In [12]:
print("\n" + "="*70)
print("🧪 EVALUATING ULTIMATE MODEL ON TEST SET")
print("="*70)

model.load_state_dict(torch.load(f"best_ultimate_{'dinov2' if USE_DINOV2 else 'maxvit'}.pth"))
model.eval()

ultimate_preds_list = []
ultimate_labs_list = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing Ultimate Model"):
        if batch is None: continue
        img, metadata, y = batch
        img, metadata = img.to(device), metadata.to(device)
        
        with autocast(device_type='cuda', dtype=torch.float16):
            logits, _ = model(img, metadata)
            probs = torch.sigmoid(logits)
        
        ultimate_preds_list.append(probs.cpu().numpy())
        ultimate_labs_list.append(y.numpy())

ultimate_preds = np.vstack(ultimate_preds_list).astype(np.float32)
ultimate_labs = np.vstack(ultimate_labs_list).astype(np.int32)

# Metrics
ultimate_auc = roc_auc_score(ultimate_labs, ultimate_preds, average='macro')
ultimate_per_class = roc_auc_score(ultimate_labs, ultimate_preds, average=None)

print(f"\n{'='*70}")
print(f"ULTIMATE MODEL RESULTS ({'DINOv2' if USE_DINOV2 else 'MaxViT'})")
print(f"{'='*70}")
print(f"Test AUC (macro): {ultimate_auc:.4f}")

print("\nPer-Class AUC:")
for i, label in enumerate(labels):
    print(f"  {label:20s}: {ultimate_per_class[i]:.4f}")

# Save
ultimate_results = {
    'model': f"Ultimate {'DINOv2' if USE_DINOV2 else 'MaxViT'} + Anatomical + Multi-Modal",
    'test_auc': ultimate_auc,
    'per_class_auc': ultimate_per_class.tolist(),
    'best_val_auc': best_auc
}

with open(f'phase7_ultimate_{"dinov2" if USE_DINOV2 else "maxvit"}_test_results.json', 'w') as f:
    json.dump(ultimate_results, f, indent=2)



🧪 EVALUATING ULTIMATE MODEL ON TEST SET


Testing Ultimate Model:   0%|          | 0/1682 [00:00<?, ?it/s]


ULTIMATE MODEL RESULTS (DINOv2)
Test AUC (macro): 0.7570

Per-Class AUC:
  Atelectasis         : 0.7139
  Consolidation       : 0.7702
  Infiltration        : 0.6687
  Pneumothorax        : 0.8201
  Edema               : 0.8689
  Emphysema           : 0.8458
  Fibrosis            : 0.7436
  Effusion            : 0.7964
  Pneumonia           : 0.6923
  Pleural_Thickening  : 0.7330
  Cardiomegaly        : 0.7793
  Nodule              : 0.6482
  Mass                : 0.6878
  Hernia              : 0.8293


In [14]:
print("\n" + "="*70)
print("🔄 REGENERATING ALL MODEL PREDICTIONS FOR 4-WAY ENSEMBLE")
print("="*70)

# Ensure test_labs exists (from ultimate model evaluation)
# If not, we already have ultimate_labs from Cell 9

test_labs = ultimate_labs  # Use the labels from ultimate model evaluation

# ============================================================
# 1. LOAD AND PREDICT: BALANCED B0
# ============================================================
print("\n1️⃣ Loading Balanced EfficientNet-B0...")

class BalancedB0(nn.Module):
    def __init__(self):
        super().__init__()
        from efficientnet_pytorch import EfficientNet
        self.net = EfficientNet.from_pretrained('efficientnet-b0')
        n = self.net._fc.in_features
        self.net._fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(n, 14))
    def forward(self, x): return self.net(x)

b0_model = BalancedB0().to(device)
b0_model.load_state_dict(torch.load('best_balanced_effnet.pth', weights_only=False))
b0_model.eval()

b0_pred_list = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="B0 Predictions"):
        if batch is None: continue
        img, metadata, y = batch  # Unpack all 3 items
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = torch.sigmoid(b0_model(img.to(device)))
        b0_pred_list.append(out.cpu().numpy())

b0_preds = np.vstack(b0_pred_list).astype(np.float32)
print(f"✅ B0 predictions shape: {b0_preds.shape}")

del b0_model
torch.cuda.empty_cache()

# ============================================================
# 2. LOAD AND PREDICT: EFFICIENTNET-B3
# ============================================================
print("\n2️⃣ Loading EfficientNet-B3...")

class EfficientNetB3(nn.Module):
    def __init__(self, num_classes=14, dropout=0.4):
        super().__init__()
        from efficientnet_pytorch import EfficientNet
        self.net = EfficientNet.from_pretrained('efficientnet-b3')
        n = self.net._fc.in_features
        self.net._fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(n, 512),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.net(x)

b3_model = EfficientNetB3().to(device)
b3_model.load_state_dict(torch.load("best_efficientnet_b3.pth"))
b3_model.eval()

b3_pred_list = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="B3 Predictions"):
        if batch is None: continue
        img, metadata, y = batch
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = torch.sigmoid(b3_model(img.to(device)))
        b3_pred_list.append(out.cpu().numpy())

test_preds = np.vstack(b3_pred_list).astype(np.float32)  # Named test_preds for consistency
print(f"✅ B3 predictions shape: {test_preds.shape}")

del b3_model
torch.cuda.empty_cache()

# ============================================================
# 3. LOAD AND PREDICT: VISION TRANSFORMER
# ============================================================
print("\n3️⃣ Loading Vision Transformer...")

class VisionTransformer(nn.Module):
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=pretrained, num_classes=0)
        self.embed_dim = self.vit.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(0.3),
            nn.Linear(self.embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        features = self.vit(x)
        return self.classifier(features)

vit_model = VisionTransformer().to(device)
vit_model.load_state_dict(torch.load("best_vit_base.pth"))
vit_model.eval()

vit_pred_list = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="ViT Predictions"):
        if batch is None: continue
        img, metadata, y = batch
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = torch.sigmoid(vit_model(img.to(device)))
        vit_pred_list.append(out.cpu().numpy())

vit_test_preds = np.vstack(vit_pred_list).astype(np.float32)
print(f"✅ ViT predictions shape: {vit_test_preds.shape}")

del vit_model
torch.cuda.empty_cache()

# ============================================================
# 4. ULTIMATE MODEL PREDICTIONS (already computed)
# ============================================================
print(f"\n4️⃣ Ultimate model predictions: {ultimate_preds.shape}")

# ============================================================
# VERIFY ALL SHAPES MATCH
# ============================================================
print("\n" + "="*70)
print("SHAPE VERIFICATION")
print("="*70)
print(f"test_labs shape:       {test_labs.shape}")
print(f"b0_preds shape:        {b0_preds.shape}")
print(f"test_preds (B3) shape: {test_preds.shape}")
print(f"vit_test_preds shape:  {vit_test_preds.shape}")
print(f"ultimate_preds shape:  {ultimate_preds.shape}")

assert b0_preds.shape == test_preds.shape == vit_test_preds.shape == ultimate_preds.shape == test_labs.shape, \
    "Shape mismatch! All prediction arrays must have the same shape."

print("\n✅ All shapes match! Ready for ensemble.")

# ============================================================
# BUILD 4-WAY ENSEMBLE
# ============================================================
print("\n" + "="*70)
print("🎯 BUILDING 4-WAY ENSEMBLE")
print("="*70)

weight_configs = [
    (0.25, 0.15, 0.25, 0.35),  # B0, B3, ViT, Ultimate
    (0.30, 0.15, 0.25, 0.30),
    (0.25, 0.10, 0.30, 0.35),
    (0.25, 0.15, 0.30, 0.30),
    (0.20, 0.10, 0.30, 0.40),  # Favor ultimate
    (0.20, 0.15, 0.25, 0.40),
    (0.15, 0.10, 0.25, 0.50),  # Heavy ultimate
]

best_4way_auc = 0
best_4way_weights = None
best_4way_preds = None

print("\nTesting 4-way weight combinations...")
for w_b0, w_b3, w_vit, w_ult in weight_configs:
    ens = w_b0 * b0_preds + w_b3 * test_preds + w_vit * vit_test_preds + w_ult * ultimate_preds
    auc_macro = roc_auc_score(test_labs, ens, average='macro')
    
    print(f"  Weights (B0={w_b0:.2f}, B3={w_b3:.2f}, ViT={w_vit:.2f}, Ultimate={w_ult:.2f}) → AUC={auc_macro:.4f}")
    
    if auc_macro > best_4way_auc:
        best_4way_auc = auc_macro
        best_4way_weights = (w_b0, w_b3, w_vit, w_ult)
        best_4way_preds = ens

print(f"\n✅ Best 4-way ensemble AUC: {best_4way_auc:.4f}")
print(f"   Optimal weights:")
print(f"     B0:      {best_4way_weights[0]:.2f}")
print(f"     B3:      {best_4way_weights[1]:.2f}")
print(f"     ViT:     {best_4way_weights[2]:.2f}")
print(f"     Ultimate: {best_4way_weights[3]:.2f}")

# Per-class analysis
final_per_class = roc_auc_score(test_labs, best_4way_preds, average=None)

print("\n" + "="*70)
print("FINAL 4-WAY ENSEMBLE: PER-CLASS RESULTS")
print("="*70)

minority_classes = ['Hernia', 'Pneumonia', 'Edema', 'Emphysema', 'Fibrosis']
for i, label in enumerate(labels):
    marker = "⭐" if label in minority_classes else "  "
    print(f"{marker} {label:20s}: {final_per_class[i]:.4f}")

minority_aucs = [final_per_class[labels.index(c)] for c in minority_classes]
print(f"\n📊 Average Minority Class AUC: {np.mean(minority_aucs):.4f}")

# Complete comparison
print("\n" + "="*70)
print("🏆 COMPLETE PERFORMANCE COMPARISON")
print("="*70)
comparison_data = {
    'Model': [
        'Baseline Ensemble',
        'Balanced B0 (Phase 4)',
        'EfficientNet-B3',
        'Vision Transformer',
        '3-Way Ensemble',
        'Ultimate Model',
        '4-Way Final Ensemble'
    ],
    'Test AUC': [
        0.7870,
        0.7945,
        0.7884,
        0.7931,
        0.8166,
        ultimate_auc,
        best_4way_auc
    ],
    'vs Baseline': [
        0.0,
        0.75,
        0.14,
        0.61,
        2.96,
        (ultimate_auc - 0.7870) * 100,
        (best_4way_auc - 0.7870) * 100
    ]
}

comp_df = pd.DataFrame(comparison_data)
print(comp_df.to_string(index=False))

print(f"\n🎉 TOTAL IMPROVEMENT: {(best_4way_auc - 0.7870)*100:+.2f}%")

# Save final results
final_results = {
    'model': '4-Way Ensemble (B0 + B3 + ViT + Ultimate)',
    'architecture': f"Ultimate: {'DINOv2' if USE_DINOV2 else 'MaxViT'} + Anatomical + Multi-Modal",
    'weights': {
        'B0': float(best_4way_weights[0]),
        'B3': float(best_4way_weights[1]),
        'ViT': float(best_4way_weights[2]),
        'Ultimate': float(best_4way_weights[3])
    },
    'test_auc_macro': float(best_4way_auc),
    'per_class_auc': final_per_class.tolist(),
    'minority_class_avg': float(np.mean(minority_aucs)),
    'component_performance': {
        'B0_alone': 0.7945,
        'B3_alone': 0.7884,
        'ViT_alone': 0.7931,
        '3-way_ensemble': 0.8166,
        'Ultimate_alone': float(ultimate_auc)
    },
    'improvement_vs_baseline': float((best_4way_auc - 0.7870) * 100)
}

with open('FINAL_4WAY_ENSEMBLE_RESULTS.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\n✅ Final results saved to FINAL_4WAY_ENSEMBLE_RESULTS.json")
print("="*70)



🔄 REGENERATING ALL MODEL PREDICTIONS FOR 4-WAY ENSEMBLE

1️⃣ Loading Balanced EfficientNet-B0...
Loaded pretrained weights for efficientnet-b0


B0 Predictions:   0%|          | 0/1682 [00:00<?, ?it/s]

✅ B0 predictions shape: (16818, 14)

2️⃣ Loading EfficientNet-B3...
Loaded pretrained weights for efficientnet-b3


B3 Predictions:   0%|          | 0/1682 [00:00<?, ?it/s]

✅ B3 predictions shape: (16818, 14)

3️⃣ Loading Vision Transformer...


ViT Predictions:   0%|          | 0/1682 [00:00<?, ?it/s]

✅ ViT predictions shape: (16818, 14)

4️⃣ Ultimate model predictions: (16818, 14)

SHAPE VERIFICATION
test_labs shape:       (16818, 14)
b0_preds shape:        (16818, 14)
test_preds (B3) shape: (16818, 14)
vit_test_preds shape:  (16818, 14)
ultimate_preds shape:  (16818, 14)

✅ All shapes match! Ready for ensemble.

🎯 BUILDING 4-WAY ENSEMBLE

Testing 4-way weight combinations...
  Weights (B0=0.25, B3=0.15, ViT=0.25, Ultimate=0.35) → AUC=0.8155
  Weights (B0=0.30, B3=0.15, ViT=0.25, Ultimate=0.30) → AUC=0.8162
  Weights (B0=0.25, B3=0.10, ViT=0.30, Ultimate=0.35) → AUC=0.8155
  Weights (B0=0.25, B3=0.15, ViT=0.30, Ultimate=0.30) → AUC=0.8166
  Weights (B0=0.20, B3=0.10, ViT=0.30, Ultimate=0.40) → AUC=0.8142
  Weights (B0=0.20, B3=0.15, ViT=0.25, Ultimate=0.40) → AUC=0.8142
  Weights (B0=0.15, B3=0.10, ViT=0.25, Ultimate=0.50) → AUC=0.8104

✅ Best 4-way ensemble AUC: 0.8166
   Optimal weights:
     B0:      0.25
     B3:      0.15
     ViT:     0.30
     Ultimate: 0.30

FINAL 4-WAY ENS